# Dark Photon Conversion Maps and Galaxy Catalogs

Visualize the spatial correlation between dark photon conversion probability and galaxy distributions. Shows how conversions preferentially occur the regions where galaxies also form.

## Setup

In [ ]:
import sys
sys.path.append("../")
sys.path.append("../21cmfast_sim/")

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab

# Jupyter setup and custom modules
%load_ext autoreload
%autoreload 2
%matplotlib inline

from plot_params import params
from galaxy_survey import *

pylab.rcParams.update(params)
cols_default = plt.rcParams['axes.prop_cycle'].by_key()['color']

## Roman Survey Setup

In [ ]:
cache_name = f'/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/v4_lightcones/'
lc_name = "LightCone_z5.5_HIIDIM=200_BOXLEN=300.0_r3843290498042390.h5"
lightconer_name = "lightconer_seed3843290498042390.pkl"

# Initialize Roman survey for Lyman-alpha emitters with pixel-based ILC cleaning
roman = Survey("Roman", "Lya", 
        cache=cache_name, 
        lc_name=lc_name, 
        lightconer_name=lightconer_name, 
        nside=2048, 
        foregrounds_path="/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_foregrounds_0.05mJy/all_foreground_maps.npy",
        use_pixel_ilc=True,  # Use pixel-based foreground cleaning
        )

## Compute Galaxy Catalog and Dark Photon Signal

In [ ]:
# Generate galaxy catalog using Lyman-alpha emitter selection criteria
roman.compute_catalog()

# Compute dark photon conversion probability for mA = 3e-14 eV
roman.compute_dp_signal(mA=3e-14)

## Visualization: Conversion Map with Galaxy Overlay

In [ ]:
gmap_indices = np.nonzero(roman.gal_map)
small_gsurvey = roman.gal_map[gmap_indices]
theta_gsurvey = roman.lat[gmap_indices]
phi_gsurvey = roman.lon[gmap_indices]
theta_gsurvey = np.repeat(theta_gsurvey, small_gsurvey)
phi_gsurvey = np.repeat(phi_gsurvey, small_gsurvey)
num_plot_gals = 10_000
subset_indices = np.random.choice(len(phi_gsurvey), replace=False, size=num_plot_gals)

In [ ]:
epsilon = 1e-6
# plt.imshow(roman.Tgamma0 * roman.Ptot[::-1,:] * ai(115) * epsilon**2, 
#         extent=[np.rad2deg(roman.lat[0,0]), np.rad2deg(roman.lat[-1,0]), np.rad2deg(roman.lat[0,0]), np.rad2deg(roman.lat[-1,0])],)
# Create conversion probability map with galaxy overlay
plt.imshow(roman.Tgamma0 * roman.Ptot[::-1,:] * ai(410) * epsilon**2,  # Conversion signal strength
            extent=[np.rad2deg(roman.lat[0,0]), np.rad2deg(roman.lat[-1,0]), 
                   np.rad2deg(roman.lat[0,0]), np.rad2deg(roman.lat[-1,0])], 
            norm="log",
            cmap="plasma")
cbar = plt.colorbar()
cbar.set_label(r"$T_{\gamma,0} P_{\gamma \to A'}$ [mK]")
plt.scatter(np.rad2deg(phi_gsurvey[subset_indices]), np.rad2deg(theta_gsurvey[subset_indices]), s=3, marker=".", c="k", alpha=0.5)

# Overlay galaxy positions as scatter points
plt.xlabel(r"Angle [$^\circ$]")
plt.ylabel(r"Angle [$^\circ$]")
plt.savefig("plots/conversion_map_gal_cat.pdf", bbox_inches="tight")